# Cassandra IoT Demo Notebook

1. Stack readiness
2. Hash ring and replication
3. Add node 4 and show redistribution
4. Schema verification
5. Query-first raw modeling
6. Sparse wide-row demo
7. Alerts and analytics
8. Room A anomaly concentration
9. Behavior profiles and ANN finale


## 0. Parameters

Fill these values before the live demo.  
Use the metadata queries below to discover good values first.


In [ ]:
from datetime import date

# Demo parameters
DATE = date.today().strftime("%Y-%m-%d")
SENSOR_UUID = "<SENSOR_UUID>"  # Replace after metadata lookup
room_A_SENSOR_UUID = "<room_A_SENSOR_UUID>"  # Optional: known anomalous sensor in room_A
ANCHOR_VECTOR = "[2150.0, 18000.0, 3.0]"   # Replace with a real profilevector if needed

2026-04-24


## 1. Stack readiness

Goal: prove that the full environment is alive before discussing data.


In [3]:
%%bash
set -e

echo "
--- nodetool status ---"
docker exec cassandra-1 nodetool status

echo "
--- kafka topics ---"
docker exec kafka kafka-topics --bootstrap-server kafka:29092 --list


--- nodetool status ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  635.62 KiB  16      76.0%             876a86f9-615d-42d9-a284-63629391b044  rack1
UN  172.18.0.2  203.95 KiB  16      64.7%             c81310fd-47fb-428b-bc4a-39f0bbf722e3  rack1
UN  172.18.0.5  129.71 KiB  16      59.3%             18c4b868-214f-48b2-8827-14f3a7e50c26  rack1


--- kafka topics ---
sensor_light
sensor_power
sensor_temp_humidity


#### Cassandra Masterless Property in action in Read Operation

Note: 
* The coordinator here does not change because we use cqlsh always on the same node (cassandra-1 here). Changing it to cassandra-2 would change the coordinator.
* Notice that, when running a query with CONSISTENCY different than ONE, the READ_REQ for the multiple nodes starts at almost the same time, so parallel request on a P2P network.
  * Example:
    * `READ_REQ message received from /172.18.0.7:7000 [Messaging-EventLoop-3-10] | 2026-04-25 07:19:10.833000 | 127.18.0.5`  
    * `READ_REQ message received from /172.18.0.7:7000 [Messaging-EventLoop-3-5] | 2026-04-25 07:19:10.834000 | 127.18.0.2`

In [ ]:
%%bash
set -e

docker exec cassandra-1 cqlsh -e "CONSISTENCY ALL; TRACING ON; SELECT * FROM iot_raw.devices_metadata WHERE sensor_type = 'power';"

Consistency level set to ALL.
TRACING set to ON

 sensor_type | sensor_id                            | description                         | location_id
-------------+--------------------------------------+-------------------------------------+-------------
       power | 01dd074c-bfb6-5c17-9c61-e8db891d5bdc |   power sensor 5 in room_C (normal) |      room_C
       power | 0883bc9f-3703-50c0-99b2-99e057fed1c5 |   power sensor 2 in room_B (normal) |      room_B
       power | 0a5cdeaf-f637-5a4f-ae7e-872c631d6fcc |   power sensor 7 in room_C (normal) |      room_C
       power | 0ed6369b-ca11-5522-b4b1-3ba59e95ee77 |   power sensor 8 in room_B (normal) |      room_B
       power | 114d73a4-8a47-55cf-a1d7-f520682421a2 | power sensor 1 in room_A (abnormal) |      room_A
       power | 125e1095-81ae-512e-812c-13f354bcca33 |   power sensor 9 in room_B (normal) |      room_B
       power | 152dde1f-7b0d-52c9-b4db-6ab29f6b43d7 |   power sensor 3 in room_C (normal) |      room_C
       power |

## 2. Hash ring, token distribution, and replication

### 2.1 Same replication strategy, same replication factor

In [9]:
%%bash
set -e

echo "--- nodetool status ---"
docker exec cassandra-1 nodetool status

echo "Owns sums to 200 because of RF=2"

echo "
--- nodetool ring ---"

docker exec cassandra-1 nodetool ring | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_raw | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_alerts | sort -k1,1

echo "
--- keyspace replication settings ---"
docker exec cassandra-1 cqlsh -e "
SELECT keyspace_name, replication
FROM system_schema.keyspaces
WHERE keyspace_name IN ('iot_raw','iot_alerts','iot_analytics')
"


--- nodetool status ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load      Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.7  3.82 MiB  16      59.3%             992a8879-14ea-44e1-828b-f16934e04f5f  rack1
UN  172.18.0.2  3.45 MiB  16      64.7%             803e3bbd-a059-4fa9-880b-aa72eb624f09  rack1
UN  172.18.0.5  4.07 MiB  16      76.0%             73373fef-78c0-48c5-a904-5cad0a041e02  rack1

Owns sums to 200 because of RF=2

--- nodetool ring ---




  
172.18.0.2       rack1       Up     Normal  3.45 MiB        64.66%              -1277325188125006372                        
172.18.0.2       rack1       Up     Normal  3.45 MiB        64.66%              1613719193259068465                         
172.18.0.2       rack1       Up     Normal  3.45 MiB        64.66%              -201569306899426833                         
172.18.0.2       rack1       Up     Normal  3.45 MiB        64.66%              -20

### 2.2 Different Replication Strategy and Factor

In [ ]:
%%bash
set -e

echo "--- ALTER KEYSPACE iot_alerts ---"
docker exec cassandra-1 cqlsh -e " ALTER KEYSPACE iot_alerts WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 3};"

echo "--- ALTER KEYSPACE iot_raw ---"
docker exec cassandra-1 cqlsh -e " ALTER KEYSPACE iot_raw WITH replication = {'class': 'NetworkTopologyStrategy', 'dc1': 2};"

SyntaxError: invalid syntax (220128806.py, line 1)

#### The instructions above change the Replication metadata of the keyspace but deos not move the data

In [ ]:
%%bash
set -e

docker exec cassandra-1 nodetool repair --full

#### Result

In [ ]:
%%bash
set -e

echo "--- nodetool status ---"
docker exec cassandra-1 nodetool status

echo "Owns sums to 200 because of RF=2"

echo "
--- nodetool ring ---"

docker exec cassandra-1 nodetool ring | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_raw | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_alerts | sort -k1,1

echo "
--- keyspace replication settings ---"
docker exec cassandra-1 cqlsh -e "
SELECT keyspace_name, replication
FROM system_schema.keyspaces
WHERE keyspace_name IN ('iot_raw','iot_alerts','iot_analytics')
"


### 2.1 Trace a real partition-key query

This query uses the human-readable partition `(location_id, date)` on `readings_by_location`.


In [12]:
from textwrap import dedent

query = dedent(
    f"""
TRACING ON;
SELECT location_id, date, timestamp, sensor_id, sensor_type
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '{DATE}'
LIMIT 5;
"""
).strip()

print(query)

TRACING ON;
SELECT location_id, date, timestamp, sensor_id, sensor_type
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '2026-04-24'
LIMIT 5;


| location_id | date | timestamp | sensor_id | sensor_type |
| :--- | :--- | :--- | :--- | :--- |
| room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 033d6766-b7b4-5754-9e28-34cccdc78f19 | temp_humidity |
| room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 066aac75-28a0-5121-9949-1f58b4b7a441 | temp_humidity |
| room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 114d73a4-8a47-55cf-a1d7-f520682421a2 | power |
| room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 1fd4a236-a4f6-51d9-ab23-16f49b2ed39d | light |
| room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 20ae2c28-545a-5bc6-aa88-81aa706481d4 | temp_humidity |

(5 rows)

Tracing session: b3a61f80-3fe7-11f1-97e0-0df443d3469a

Execute CQL3 query | 2026-04-24 14:13:00.920000 | 172.18.0.2 | 0 | 127.0.0.1

Parsing SELECT location_id, date, timestamp, sensor_id, sensor_type\nFROM iot_raw.readings_by_location\nWHERE location_id = 'room_A' AND date = '2026-04-24'\nLIMIT 5; 

[Native-Transport-Requests-1] | 2026-04-24 14:13:00.920001 | 172.18.0.2 | 133 | 127.0.0.1

Preparing statement [Native-Transport-Requests-1] | 2026-04-24 14:13:00.920002 | 172.18.0.2 | 269 | 127.0.0.1

reading data from /172.18.0.6:7000 [Native-Transport-Requests-1] | 2026-04-24 14:13:00.921000 | 172.18.0.2 | 915 | 127.0.0.1

Sending READ_REQ message to /172.18.0.6:7000 message size 179 bytes [Messaging-EventLoop-3-9] | 2026-04-24 14:13:00.921001 | 172.18.0.2 | 1088 | 127.0.0.1

READ_REQ message received from /172.18.0.2:7000 [Messaging-EventLoop-3-9] | 2026-04-24 14:13:00.922000 | 172.18.0.6 | 118 | 127.0.0.1

Executing single-partition query on readings_by_location [ReadStage-3] | 2026-04-24 14:13:00.923000 | 172.18.0.6 | 1106 | 127.0.0.1

Acquiring sstable references [ReadStage-3] | 2026-04-24 14:13:00.923001 | 172.18.0.6 | 1162 | 127.0.0.1

Skipped 0/1 non-slice-intersecting sstables, included 0 due to tombstones [ReadStage-3] | 2026-04-24 14:13:00.923002 | 172.18.0.6 | 1246 | 127.0.0.1

Partition index found for sstable 1, size = 0 [ReadStage-3] | 2026-04-24 14:13:00.924000 | 172.18.0.6 | 1548 | 127.0.0.1

Merged data from memtables and 1 sstables [ReadStage-3] | 2026-04-24 14:13:00.924001 | 172.18.0.6 | 2227 | 127.0.0.1

Read 5 live rows and 0 tombstone cells [ReadStage-3] | 2026-04-24 14:13:00.924002 | 172.18.0.6 | 2301 | 127.0.0.1

Enqueuing response to /172.18.0.2:7000 [ReadStage-3] | 2026-04-24 14:13:00.924003 | 172.18.0.6 | 2343 | 127.0.0.1

READ_RSP message received from /172.18.0.6:7000 [Messaging-EventLoop-3-12] | 2026-04-24 14:13:00.925000 | 172.18.0.2 | 4753 | 127.0.0.1

Sending READ_RSP message to cassandra-1/172.18.0.2:7000 message size 295 bytes [Messaging-EventLoop-3-1] | 2026-04-24 14:13:00.925000 | 172.18.0.6 | 2483 | 127.0.0.1

Processing response from /172.18.0.6:7000 [RequestResponseStage-2] | 2026-04-24 14:13:00.925001 | 172.18.0.2 | 4854 | 127.0.0.1

Request complete | 2026-04-24 14:13:00.925133 | 172.18.0.2 | 5133 | 127.0.0.1

In [14]:
%%bash
set -e
DATE_VALUE="{DATE}"

docker exec cassandra-1 cqlsh -e "
TRACING ON;
SELECT location_id, date, timestamp, sensor_id, sensor_type
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '${DATE_VALUE}'
LIMIT 5;
"

<stdin>:1:InvalidRequest: Error from server: code=2200 [Invalid query] message="Unable to coerce '{DATE}' to a formatted date (long)"
<stdin>:1:SyntaxException: line 1:0 no viable alternative at input ';' ([;])


TRACING set to ON


CalledProcessError: Command 'b'set -e\nDATE_VALUE="{DATE}"\n\ndocker exec cassandra-1 cqlsh -e "\nTRACING ON;\nSELECT location_id, date, timestamp, sensor_id, sensor_type\nFROM iot_raw.readings_by_location\nWHERE location_id = \'room_A\' AND date = \'${DATE_VALUE}\'\nLIMIT 5;\n"\n'' returned non-zero exit status 2.

### 2.2 Optional: direct endpoint lookup

If supported by your Cassandra image, this is the cleanest replica demonstration.


In [32]:
print(f"docker exec cassandra-1 nodetool getendpoints iot_raw readings_by_location 'room_A:{DATE}'")

docker exec cassandra-1 nodetool getendpoints iot_raw readings_by_location 'room_A:2026-04-24'


In [33]:
!docker exec cassandra-1 nodetool getendpoints iot_raw readings_by_location 'room_A:2026-04-24'

172.18.0.6
172.18.0.7


### 2.3 Partition size balance check (`nodetool tablestats`)

This command prints table-level storage statistics, including **minimum** and **maximum partition bytes**.  


In [40]:
!docker exec cassandra-1 nodetool tablestats iot_raw.devices_metadata

Total number of tables: 1
----------------
Keyspace: iot_raw
	Read Count: 0
	Read Latency: NaN ms
	Write Count: 523
	Write Latency: 2.7103499043977055 ms
	Pending Flushes: 0
		Table: devices_metadata
		SSTable count: 1
		Old SSTable count: 0
		Max SSTable size: 8.009KiB
		SSTables in each level: [1, 0, 0, 0, 0, 0, 0, 0, 0]
		SSTable bytes in each level: [3003, 0, 0, 0, 0, 0, 0, 0, 0]
		Space used (live): 8201
		Space used (total): 8201
		Space used by snapshots (total): 0
		Off heap memory used (total): 33
		SSTable Compression Ratio: 0.47247
		Number of partitions (estimate): 3
		Memtable cell count: 0
		Memtable data size: 0
		Memtable off heap memory used: 0
		Memtable switch count: 1
		Speculative retries: 0
		Local read count: 0
		Local read latency: NaN ms
		Local write count: 166
		Local write latency: NaN ms
		Local read/write ratio: 0.00000
		Pending flushes: 0
		Percent repaired: 0.0
		Bytes repaired: 0B
		Bytes unrepaired: 6.207KiB
		Bytes pending repair: 0B
		Bloom filter f

## 3. Elasticity demo: add node 4 and show redistribution

Goal: show that Cassandra can rebalance token ownership when a new node joins.

> This section assumes you added a `cassandra-4` service to `docker-compose.yml` beforehand.
> If not, treat this section as a scripted terminal demo template.


In [17]:
%%bash
set -e

echo "--- BEFORE node 4 ---"
docker exec cassandra-1 nodetool status
docker exec cassandra-1 nodetool ring iot_raw

docker compose up -d cassandra-4

--- BEFORE node 4 ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  249.47 KiB  16      50.7%             876a86f9-615d-42d9-a284-63629391b044  rack1
UN  172.18.0.7  187.25 KiB  16      52.0%             6bf829d5-eb63-4c9e-9060-352f78fe64b6  rack1
UN  172.18.0.2  215.77 KiB  16      48.5%             c81310fd-47fb-428b-bc4a-39f0bbf722e3  rack1
UN  172.18.0.5  212.92 KiB  16      48.8%             18c4b868-214f-48b2-8827-14f3a7e50c26  rack1


Datacenter: dc1
Address          Rack        Status State   Load            Owns                Token                                       
                                                                                8931197532403896265                         
172.18.0.6       rack1       Up     Normal  249.47 KiB      50.68%              -9084535060374987267                        
172.18.0.7       rack1   

In [18]:
%%bash
set -e

echo "--- AFTER node 4 joins ---"
docker exec cassandra-1 nodetool status
docker exec cassandra-1 nodetool ring iot_raw

--- AFTER node 4 joins ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  249.47 KiB  16      50.7%             876a86f9-615d-42d9-a284-63629391b044  rack1
UN  172.18.0.7  152.69 KiB  16      52.0%             6bf829d5-eb63-4c9e-9060-352f78fe64b6  rack1
UN  172.18.0.2  196.46 KiB  16      48.5%             c81310fd-47fb-428b-bc4a-39f0bbf722e3  rack1
UN  172.18.0.5  212.92 KiB  16      48.8%             18c4b868-214f-48b2-8827-14f3a7e50c26  rack1


Datacenter: dc1
Address          Rack        Status State   Load            Owns                Token                                       
                                                                                8931197532403896265                         
172.18.0.6       rack1       Up     Normal  249.47 KiB      50.68%              -9084535060374987267                        
172.18.0.7       rac

## 4. Schema verification and Cassandra data model

Goal: verify the real keyspaces and table names before using them live.


In [27]:
%bash
set -e

docker exec cassandra-1 cqlsh -e "DESCRIBE KEYSPACES;"

echo "
--- iot_raw tables ---"
docker exec cassandra-1 cqlsh -e "DESCRIBE TABLES IN iot_raw;"

echo "
--- iot_alerts tables ---"
docker exec cassandra-1 cqlsh -e "DESCRIBE TABLES IN iot_alerts;"

echo "
--- iot_analytics tables ---"
docker exec cassandra-1 cqlsh -e "DESCRIBE TABLES IN iot_analytics;"

echo "
--- system_schema.tables ---"
docker exec cassandra-1 cqlsh -e "
SELECT keyspace_name, table_name
FROM system_schema.tables
WHERE keyspace_name IN ('iot_raw','iot_alerts','iot_analytics');
"


SyntaxError: unterminated string literal (detected at line 6) (3901566420.py, line 6)

## 5. Query-first modeling through raw tables

Goal: show that multiple raw tables exist because they serve different access patterns.


In [ ]:
%%bash
set -e

echo "--- metadata: temphumidity ---"
docker exec cassandra-1 cqlsh -e "
SELECT sensor_type, sensor_id, location_id, description
FROM iot_raw.devicesmetadata
WHERE sensor_type = 'temphumidity'
LIMIT 5;
"

echo "
--- metadata: power ---"
docker exec cassandra-1 cqlsh -e "
SELECT sensor_type, sensor_id, location_id, description
FROM iot_raw.devicesmetadata
WHERE sensor_type = 'power'
LIMIT 5;
"


In [ ]:
from textwrap import dedent
raw_queries = {
    'temphumiditybysensor': dedent(f'''
        SELECT sensor_id, date, timestamp, location_id, temperature, humidity
        FROM iot_raw.temphumiditybysensor
        WHERE sensor_id = {SENSOR_UUID} AND date = '{DATE}'
        LIMIT 5;
    ''').strip(),
    'powerbysensor': dedent(f'''
        SELECT sensor_id, date, timestamp, location_id, voltage, amperage, wattage
        FROM iot_raw.powerbysensor
        WHERE sensor_id = {SENSOR_UUID} AND date = '{DATE}'
        LIMIT 5;
    ''').strip(),
}
for name, q in raw_queries.items():
    print(f"--- {name} ---
{q}
")


In [ ]:
%%bash
set -e

echo "Replace SENSOR_UUID and DATE in the next two commands before executing them live:"
echo
cat <<'SQL'
SELECT sensor_id, date, timestamp, location_id, temperature, humidity
FROM iot_raw.temphumiditybysensor
WHERE sensor_id = <SENSOR_UUID> AND date = '<DATE>'
LIMIT 5;
SQL

echo
cat <<'SQL'
SELECT sensor_id, date, timestamp, location_id, voltage, amperage, wattage
FROM iot_raw.powerbysensor
WHERE sensor_id = <SENSOR_UUID> AND date = '<DATE>'
LIMIT 5;
SQL


## 6. Sparse wide-row demo with `readings_by_location`

Goal: show that the location table stores heterogeneous readings and only fills relevant columns per sensor type.


In [45]:
from textwrap import dedent
sparse_query = dedent(f'''
SELECT location_id, date, timestamp, sensor_id, sensor_type,
       temperature, humidity, light_level, amperage, voltage, wattage
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '{DATE}'
LIMIT 12;
''').strip()
print(sparse_query)


SELECT location_id, date, timestamp, sensor_id, sensor_type,
       temperature, humidity, light_level, amperage, voltage, wattage
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '2026-04-24'
LIMIT 12;


In [46]:
%%bash
set -e
DATE_VALUE={DATE}

docker exec cassandra-1 cqlsh -e "
SELECT location_id, date, timestamp, sensor_id, sensor_type,
       temperature, humidity, light_level, amperage, voltage, wattage
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '2026-04-24'
LIMIT 12
"



 location_id | date       | timestamp                       | sensor_id                            | sensor_type   | temperature | humidity | light_level | amperage | voltage | wattage
-------------+------------+---------------------------------+--------------------------------------+---------------+-------------+----------+-------------+----------+---------+------------
      room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 033d6766-b7b4-5754-9e28-34cccdc78f19 | temp_humidity |        19.8 |    44.79 |        null |     null |    null |       null
      room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 066aac75-28a0-5121-9949-1f58b4b7a441 | temp_humidity |       25.33 |    49.66 |        null |     null |    null |       null
      room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 | 114d73a4-8a47-55cf-a1d7-f520682421a2 |         power |        null |     null |        null |     1.88 |  232.06 |   436.2728
      room_A | 2026-04-24 | 2026-04-24 14:04:34.000000+0000 |

## 7. Alerts and 30-second analytics

Goal: show derived operational outputs written by Spark into Cassandra.


In [ ]:
%%bash
set -e

echo "Replace SENSOR_UUID and DATE in the next commands before running them live."
echo
cat <<'SQL'
SELECT sensor_id, timestamp, alertid, location_id, alerttype, severity, alertmessage
FROM iotalerts.sensoralerts
WHERE sensor_id = <SENSOR_UUID>
LIMIT 10;
SQL

echo
cat <<'SQL'
SELECT sensor_id, date, windowstart, sensor_type, location_id,
       avgvalue, minvalue, maxvalue, readingcount
FROM iotanalytics.sensoraggregates30s
WHERE sensor_id = <SENSOR_UUID> AND date = '<DATE>'
LIMIT 10;
SQL

echo
cat <<'SQL'
SELECT sensor_type, date, windowstart, sensor_id, location_id, avgvalue
FROM iotanalytics.aggregatesbytype
WHERE sensor_type = 'power' AND date = '<DATE>'
LIMIT 10;
SQL


## 8. Room A anomaly concentration

Goal: show that the simulated anomalies cluster in `room_A`, matching the project narrative.


In [ ]:
%%bash
set -e

echo "--- Optional broad inspection (small demo dataset only) ---"
cat <<'SQL'
SELECT location_id, timestamp, sensor_id, alerttype, severity
FROM iotalerts.sensoralerts
ALLOW FILTERING;
SQL

echo

echo "--- Safer focused version: use a known room_A sensor ---"
cat <<'SQL'
SELECT sensor_id, timestamp, alerttype, severity, alertmessage
FROM iotalerts.sensoralerts
WHERE sensor_id = <room_A_SENSOR_UUID>
LIMIT 10;
SQL


## 9. Behavior profiles and Cassandra 5 ANN finale

Goal: end with the advanced Cassandra 5 vector-search demo inside a meaningful `(location_id, sensor_type)` partition.


In [ ]:
%%bash
set -e

echo "--- inspect available profiles in room_A/power ---"
docker exec cassandra-1 cqlsh -e "
SELECT location_id, sensor_type, sensor_id, lastupdatedat,
       profilesize, meanvalue, variancevalue, spikecount, profilevector
FROM iotanalytics.sensorbehaviorprofiles
WHERE location_id = 'room_A' AND sensor_type = 'power'
LIMIT 10;
"


In [ ]:
%%bash
set -e

echo "Replace SENSOR_UUID or paste a real anchor UUID from the previous result:"
cat <<'SQL'
SELECT sensor_id, profilevector, meanvalue, variancevalue, spikecount
FROM iotanalytics.sensorbehaviorprofiles
WHERE location_id = 'room_A' AND sensor_type = 'power' AND sensor_id = <SENSOR_UUID>;
SQL

echo

echo "Replace the ANN vector with a real profilevector if you want the cleanest demo:"
cat <<'SQL'
SELECT sensor_id, meanvalue, variancevalue, spikecount, lastupdatedat
FROM iotanalytics.sensorbehaviorprofiles
WHERE location_id = 'room_A' AND sensor_type = 'power'
ORDER BY profilevector ANN OF [2150.0, 18000.0, 3.0]
LIMIT 5;
SQL


## 10. Optional helper notes for live delivery

- Prefer running the notebook in **read-only narrative mode** and executing only the cells you really need live.
- For the ring demo, rehearse the node-4 bootstrap beforehand.
- For the ANN demo, prepare one known `sensor_id` and one known real `profilevector` in advance.
- If a command is risky or slow, keep it in the notebook as a reference cell and run it manually from the terminal instead.
